<a href="https://colab.research.google.com/github/sairas2124/Gradient_descent-/blob/main/pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.__version__)

2.10.0+cu128


In [ ]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [ ]:
z = w*x+b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [ ]:
y_pred= torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [ ]:
import torch.nn.functional as F
loss = F.binary_cross_entropy(y_pred,y)
loss

tensor(6.7012, grad_fn=<BinaryCrossEntropyBackward0>)

In [ ]:
loss.backward()

In [ ]:
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


Training pipeline in pytorch


---



In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [ ]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [ ]:
df.shape

(569, 31)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_train

array([[ 2.04959221, -0.9909777 ,  2.02853779, ...,  1.27826462,
        -0.27777877,  0.15701178],
       [-0.53833995, -1.40644744, -0.51925916, ..., -0.48405285,
         0.95558921,  0.47892112],
       [ 1.49422986,  0.93716918,  1.52896976, ...,  0.66076996,
         0.03932531, -0.43194235],
       ...,
       [ 1.32790307,  0.12918382,  1.19319453, ...,  0.24603474,
        -0.5498207 , -1.48276641],
       [ 1.4322097 ,  0.70533248,  1.46345264, ...,  0.88196207,
         0.4949538 ,  0.46772428],
       [-0.18595268,  0.4964499 , -0.22852695, ..., -0.03352752,
         0.162829  , -0.69394856]])

In [ ]:
y_train

,diagnosis
372,M
130,B
244,M
559,B
246,B
...,...
404,B
470,B
277,M
198,M


In [ ]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [ ]:
X_train_tensor.shape

torch.Size([455, 30])

In [ ]:
y_train_tensor.shape

torch.Size([455])

In [ ]:
class MySimple():
  def __init__(self,X):
    self.weight = torch.rand(X.shape[1], 1, dtype= torch.float64, requires_grad=True)
    self.bias = torch.rand(1, dtype= torch.float64, requires_grad=True)

  def forward(self, X):
    z = torch.matmul(X, self.weight) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
    loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
    return loss


In [ ]:
learning_rate = 0.1
epochs = 25

Training pipeline

In [ ]:
# create model
model = MySimple(X_train_tensor)


for epoch in range(epochs):

  # forward pass
  y_pred = model.forward(X_train_tensor)

  # loss calculate
  loss = model.loss_function(y_pred, y_train_tensor)

  # backward pass
  loss.backward()

  # parameters update
  with torch.no_grad():
    model.weight -= learning_rate * model.weight.grad
    model.bias -= learning_rate * model.bias.grad

  # zero gradients
  model.weight.grad.zero_()
  model.bias.grad.zero_()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 3.7206699084176593
Epoch: 2, Loss: 3.6089880661005336
Epoch: 3, Loss: 3.49428014576169
Epoch: 4, Loss: 3.3775865413883057
Epoch: 5, Loss: 3.257280393880784
Epoch: 6, Loss: 3.1304039821964422
Epoch: 7, Loss: 2.996724918924084
Epoch: 8, Loss: 2.8619139073112603
Epoch: 9, Loss: 2.7261766475100675
Epoch: 10, Loss: 2.582200719206499
Epoch: 11, Loss: 2.4332979387243014
Epoch: 12, Loss: 2.2798756486316996
Epoch: 13, Loss: 2.1234719655240375
Epoch: 14, Loss: 1.973909099011897
Epoch: 15, Loss: 1.8314466250148118
Epoch: 16, Loss: 1.6957712071665805
Epoch: 17, Loss: 1.570568337509646
Epoch: 18, Loss: 1.455478448317513
Epoch: 19, Loss: 1.3522673291882357
Epoch: 20, Loss: 1.257120106513054
Epoch: 21, Loss: 1.17418489649543
Epoch: 22, Loss: 1.1054989909769835
Epoch: 23, Loss: 1.049646813347801
Epoch: 24, Loss: 1.00490839397291
Epoch: 25, Loss: 0.9694393644303968


In [ ]:
model.bias

tensor([0.3497], dtype=torch.float64, requires_grad=True)

In [ ]:
# model evaluation
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')


Accuracy: 0.5794090628623962


Advanced Neural Network

In [4]:
#create model class
import torch
import torch.nn as nn

class Model(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    out = self.linear(features)
    out = self.sigmoid(out)
    return out

In [5]:
features = torch.rand(10,5)

model = Model(features.shape[1])
print(model(features))

tensor([[0.4130],
        [0.4139],
        [0.4293],
        [0.3778],
        [0.4046],
        [0.4125],
        [0.3891],
        [0.4183],
        [0.4157],
        [0.4015]], grad_fn=<SigmoidBackward0>)
